In [35]:
# Colab cell (code)
!pip install --quiet flask flask_sqlalchemy pyngrok qrcode[pil] pandas pillow

In [36]:
# Colab cell (code)
from getpass import getpass
from pyngrok import conf

ngrok_token = getpass("Enter your ngrok authtoken (leave blank to use default free tunnels): ")
if ngrok_token:
    conf.get_default().auth_token = ngrok_token
    print("ngrok auth token set.")
else:
    print("No token provided — using public/free ngrok tunnels (rate-limited).")

Enter your ngrok authtoken (leave blank to use default free tunnels): ··········
ngrok auth token set.


In [37]:
# Colab cell (code)
# Paste/Run the full app code below

from flask import Flask, request, redirect, render_template_string, send_file, Response
from flask_sqlalchemy import SQLAlchemy
import string, random, datetime, io, base64
import pandas as pd
import qrcode
from urllib.parse import urlparse

app = Flask(__name__)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///url.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
db = SQLAlchemy(app)

class URL(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    long_url = db.Column(db.String(1000), nullable=False)
    short_code = db.Column(db.String(8), unique=True, nullable=False)
    title = db.Column(db.String(200), nullable=True)
    description = db.Column(db.String(1000), nullable=True)
    tags = db.Column(db.String(200), nullable=True)
    clicks = db.Column(db.Integer, default=0)
    created_at = db.Column(db.DateTime, default=datetime.datetime.utcnow)

def generate_short_code(length=6):
    chars = string.ascii_letters + string.digits
    while True:
        code = ''.join(random.choices(chars, k=length))
        if not URL.query.filter_by(short_code=code).first():
            return code

def is_valid_url(url):
    try:
        result = urlparse(url)
        return result.scheme in ("http", "https") and result.netloc != ""
    except Exception:
        return False

def make_qr_data_uri(url):
    qr = qrcode.QRCode(box_size=4, border=2)
    qr.add_data(url)
    qr.make(fit=True)
    img = qr.make_image(fill_color="black", back_color="white")
    bio = io.BytesIO()
    img.save(bio, format='PNG')
    bio.seek(0)
    data = base64.b64encode(bio.read()).decode('utf-8')
    return f"data:image/png;base64,{data}"

HTML_TEMPLATE = '''
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>Student Link Manager & URL Shortener</title>
  <style>
    body { font-family: Arial, sans-serif; max-width:900px; margin:2rem auto; }
    input, textarea { width:100%; padding:8px; margin:6px 0; }
    table { width:100%; border-collapse:collapse; margin-top:1rem }
    th, td { padding:8px; border:1px solid #ddd }
    .small { font-size:0.9rem; color:#555 }
    .actions { display:flex; gap:6px }
    button.copy { padding:6px 10px }
  </style>
</head>
<body>
  <h1>📚 Student Link Manager + URL Shortener</h1>
  <p class="small">Save resources, tags, notes, export CSV, and generate QR codes.</p>

  <h2>Add / Shorten a Link</h2>
  <form method="POST" action="/add">
    <input name="long_url" placeholder="Full URL (include http:// or https://)" required>
    <input name="title" placeholder="Title (e.g., Lecture 3: Recursion)">
    <textarea name="description" placeholder="Short description / notes (optional)" rows="2"></textarea>
    <input name="tags" placeholder="Tags (comma-separated, e.g., algorithms,python)">
    <label>Custom short code (optional, alphanum, max 8 chars): <input name="custom_code" style="width:200px"></label>
    <button type="submit">Save & Shorten</button>
  </form>

  <h2>Search / Filter</h2>
  <form method="GET" action="/search">
    <input name="q" placeholder="Search by title, description, tags or URL" value="{{ q|default('') }}">
    <button type="submit">Search</button>
    <a href="/export">Export CSV</a>
    <a href="/">Clear</a>
  </form>

  {% if short_url %}
    <h3>Short URL</h3>
    <p><a href="{{ short_url }}" target="_blank">{{ short_url }}</a></p>
    <div class="actions">
      <button class="copy" onclick="navigator.clipboard.writeText('{{ short_url }}')">Copy Short URL</button>
    </div>
    <p>QR code:</p>
    <img src="{{ qr_data_uri }}" alt="QR code">
  {% endif %}

  <h2>Recent / Matching Links</h2>
  <table>
    <thead><tr><th>Title</th><th>Short</th><th>Clicks</th><th>Tags</th><th>Notes</th><th>Created</th></tr></thead>
    <tbody>
    {% for u in urls %}
      <tr>
        <td><b>{{ u.title or u.long_url[:60] }}</b></td>
        <td>
          <a href="{{ request.host_url }}{{ u.short_code }}" target="_blank">{{ request.host_url }}{{ u.short_code }}</a>
          <div class="small">Orig: <a href="{{ u.long_url }}" target="_blank">{{ u.long_url|truncate(60) }}</a></div>
        </td>
        <td>{{ u.clicks }}</td>
        <td>{{ u.tags or '' }}</td>
        <td>{{ u.description or '' }}</td>
        <td class="small">{{ u.created_at.strftime('%Y-%m-%d %H:%M') }}</td>
      </tr>
    {% endfor %}
    </tbody>
  </table>

  <hr>
  <h3>Quick Tips for Students</h3>
  <ul>
    <li>Use tags to group resources by course or topic (e.g., cs101, ml).</li>
    <li>Add a short description or lecture notes so you remember why you saved the link.</li>
    <li>Export your links as CSV before exams or to share with classmates.</li>
    <li>Use the QR code to quickly open links on your phone during study sessions.</li>
  </ul>

</body>
</html>
'''

@app.route('/', methods=['GET'])
def index():
    urls = URL.query.order_by(URL.created_at.desc()).limit(25).all()
    return render_template_string(HTML_TEMPLATE, urls=urls)

@app.route('/add', methods=['POST'])
def add_link():
    long_url = request.form.get('long_url')
    title = request.form.get('title')
    description = request.form.get('description')
    tags = request.form.get('tags')
    custom_code = request.form.get('custom_code')

    if not long_url or not is_valid_url(long_url):
        return "Invalid URL. Make sure you include http:// or https://", 400

    existing = URL.query.filter_by(long_url=long_url).first()
    if existing:
        url_entry = existing
    else:
        if custom_code:
            if URL.query.filter_by(short_code=custom_code).first():
                return f"Custom code '{custom_code}' is already taken.", 400
            short_code = custom_code
        else:
            short_code = generate_short_code()
        url_entry = URL(long_url=long_url, short_code=short_code, title=title, description=description, tags=tags)
        db.session.add(url_entry)
        db.session.commit()

    short_url = request.host_url + url_entry.short_code
    qr_data = make_qr_data_uri(short_url)
    urls = URL.query.order_by(URL.created_at.desc()).limit(25).all()
    return render_template_string(HTML_TEMPLATE, short_url=short_url, qr_data_uri=qr_data, urls=urls)

@app.route('/search', methods=['GET'])
def search():
    q = request.args.get('q', '').strip()
    if not q:
        return redirect('/')
    like = f"%{q}%"
    results = URL.query.filter(
        (URL.title.ilike(like)) |
        (URL.description.ilike(like)) |
        (URL.tags.ilike(like)) |
        (URL.long_url.ilike(like))
    ).order_by(URL.created_at.desc()).all()
    return render_template_string(HTML_TEMPLATE, urls=results, q=q)

@app.route('/export')
def export_csv():
    all_urls = URL.query.order_by(URL.created_at.desc()).all()
    data = [
        {
            'title': u.title,
            'long_url': u.long_url,
            'short_url': request.host_url + u.short_code,
            'tags': u.tags,
            'description': u.description,
            'clicks': u.clicks,
            'created_at': u.created_at.strftime('%Y-%m-%d %H:%M:%S')
        }
        for u in all_urls
    ]
    df = pd.DataFrame(data)
    csv_io = io.StringIO()
    df.to_csv(csv_io, index=False)
    csv_io.seek(0)
    return Response(
        csv_io.getvalue(),
        mimetype='text/csv',
        headers={"Content-disposition": "attachment; filename=links_export.csv"}
    )

@app.route('/qr/<short_code>')
def qr(short_code):
    url_entry = URL.query.filter_by(short_code=short_code).first_or_404()
    short_url = request.host_url + url_entry.short_code
    qr_img = qrcode.make(short_url)
    bio = io.BytesIO()
    qr_img.save(bio, format='PNG')
    bio.seek(0)
    return send_file(bio, mimetype='image/png')

@app.route('/<short_code>')
def redirect_short_url(short_code):
    url_entry = URL.query.filter_by(short_code=short_code).first_or_404()
    try:
        url_entry.clicks = (url_entry.clicks or 0) + 1
        db.session.commit()
    except Exception:
        db.session.rollback()
    return redirect(url_entry.long_url)

# CLI helpers
def list_urls(limit=50):
    return URL.query.order_by(URL.created_at.desc()).limit(limit).all()

def clear_all(confirm=False):
    if not confirm:
        raise RuntimeError('Pass confirm=True to actually clear all entries')
    num = URL.query.delete()
    db.session.commit()
    return num

# Create DB schema
with app.app_context():
    db.create_all()

print("Flask app code loaded. Ready to start server (next cell).")

Flask app code loaded. Ready to start server (next cell).


In [38]:
# show processes using port 5000 (Colab / Linux)
!lsof -i:5000 -P -n || true

# alternative using ss:
!ss -ltnp | grep ':5000' || true

# or using netstat:
!netstat -ltnp 2>/dev/null | grep ':5000' || true

In [39]:
# Graceful kill (safer)
!kill $(lsof -t -i:5000) 2>/dev/null || true
# If still alive, force kill:
!kill -9 $(lsof -t -i:5000) 2>/dev/null || true

In [40]:
!fuser -k 5000/tcp 2>/dev/null || true

In [41]:
from pyngrok import ngrok
# disconnect all tunnels known to this client
for t in ngrok.get_tunnels():
    try:
        ngrok.disconnect(t.public_url)
    except Exception:
        pass
# ensure any ngrok process is killed
ngrok.kill()
print("ngrok tunnels stopped")

ngrok tunnels stopped


In [42]:
# Combined cleanup + optional restart helper for Colab
START_FLASK = False   # set True to auto-restart Flask if your `app` variable exists in notebook
FLASK_PORT = 5000

import time, sys
from pyngrok import ngrok
import subprocess, os

print("=== Processes using port", FLASK_PORT, "===")
!lsof -i:{FLASK_PORT} -P -n || true

# Attempt to kill processes using the port
print("\nKilling processes on port", FLASK_PORT, "...")
!kill -9 $(lsof -t -i:{FLASK_PORT}) 2>/dev/null || true
time.sleep(1)

# Stop pyngrok tunnels/processes
try:
    ngrok.kill()
    print("ngrok.kill() called — stopped pyngrok-managed tunnels/processes.")
except Exception as e:
    print("ngrok.kill() error (may be fine):", e)

time.sleep(1)
print("\nAfter cleanup, processes using port", FLASK_PORT, ":")
!lsof -i:{FLASK_PORT} -P -n || true

# Optional: start flask if requested and app is defined in notebook
if START_FLASK:
    if 'app' not in globals():
        print("\nERROR: Flask `app` object not found in globals. Define your app before using START_FLASK=True.")
        sys.exit(1)

    import threading
    def run_flask():
        # ensure reloader is off to avoid double-start in notebooks
        app.run(port=FLASK_PORT, host="127.0.0.1", debug=False, use_reloader=False)

    thread = threading.Thread(target=run_flask, daemon=True)
    thread.start()
    time.sleep(2)
    print("\nFlask should be running on port", FLASK_PORT)
    !lsof -i:{FLASK_PORT} -P -n || true

    try:
        public_url = ngrok.connect(FLASK_PORT)
        print("\nngrok tunnel:", public_url)
    except Exception as e:
        print("\nFailed to open ngrok tunnel:", e)
        print("If this is an ngrok auth/session error, visit dashboard.ngrok.com/agents and kill other sessions or revoke the token.")

=== Processes using port 5000 ===

Killing processes on port 5000 ...
ngrok.kill() called — stopped pyngrok-managed tunnels/processes.

After cleanup, processes using port 5000 :


In [43]:
# Colab cell (code)
import threading, time
from pyngrok import ngrok

def run_flask():
    # set host to 0.0.0.0 so ngrok can connect; keep debug off in Colab
    app.run(port=5000, host="127.0.0.1")

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

# wait a moment for server to start
time.sleep(2)

# open ngrok tunnel to port 5000
public_url = ngrok.connect(5000)
print("🚀 Your app is live at:", public_url)
print("Visit the printed URL in a browser to use the app.")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 Your app is live at: NgrokTunnel: "https://f9d5-34-12-218-71.ngrok-free.app" -> "http://localhost:5000"
Visit the printed URL in a browser to use the app.
